<a href="https://colab.research.google.com/github/nguyenus513/hadoop-lab/blob/main/Lab03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark import SparkContext
import datetime

sc = SparkContext("local", "BaiTap")
sc.setLogLevel("ERROR")
print("SparkContext đã khởi động!")

SparkContext đã khởi động!


## Bài 1: Tính điểm trung bình và tổng số lượt đánh giá cho mỗi phim

In [4]:
# Bước 1: Đọc movies.txt → tạo dict {MovieID: Title}
movies_rdd = sc.textFile("movies.txt")
movie_map = movies_rdd.map(lambda line: line.split(",")) \
                       .map(lambda x: (x[0], x[1])) \
                       .collectAsMap()  # {MovieID: Title}

# Bước 2: Đọc cả 2 file ratings, gộp lại
ratings_rdd = sc.textFile("ratings_1.txt,ratings_2.txt")

# Bước 3: Map → (MovieID, (rating, 1))
movie_rating = ratings_rdd.map(lambda line: line.split(",")) \
                           .map(lambda x: (x[1], (float(x[2]), 1)))

# Bước 4: Reduce → tính tổng điểm và số lượt
movie_sum = movie_rating.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))

# Bước 5: Tính trung bình, lọc phim >= 5 lượt đánh giá
movie_avg = movie_sum.mapValues(lambda x: (round(x[0]/x[1], 2), x[1])) \
                     .filter(lambda x: x[1][1] >= 5)

# Bước 6: Tìm phim có điểm cao nhất
best_movie_id, (best_avg, best_count) = movie_avg.max(key=lambda x: x[1][0])
best_title = movie_map.get(best_movie_id, "Unknown")

print("=== Kết quả Bài 1 ===")
print("\nTop phim theo điểm trung bình (có >= 5 lượt đánh giá):")
results = movie_avg.map(lambda x: (movie_map.get(x[0], "Unknown"), x[1][0], x[1][1])) \
                   .sortBy(lambda x: -x[1]) \
                   .take(10)
for title, avg, count in results:
    print(f"  {title}: avg={avg}, votes={count}")

print(f"\n>>> Phim có điểm TB cao nhất: {best_title} ({best_avg} sao, {best_count} lượt)")

=== Kết quả Bài 1 ===

Top phim theo điểm trung bình (có >= 5 lượt đánh giá):
  Sunset Boulevard (1950): avg=4.36, votes=7
  The Terminator (1984): avg=4.06, votes=18
  The Godfather: Part II (1974): avg=4.0, votes=17
  The Lord of the Rings: The Fellowship of the Ring (2001): avg=3.89, votes=18
  No Country for Old Men (2007): avg=3.89, votes=18
  The Social Network (2010): avg=3.86, votes=7
  The Lord of the Rings: The Return of the King (2003): avg=3.82, votes=11
  E.T. the Extra-Terrestrial (1982): avg=3.67, votes=18
  Gladiator (2000): avg=3.61, votes=18
  Fight Club (1999): avg=3.5, votes=7

>>> Phim có điểm TB cao nhất: Sunset Boulevard (1950) (4.36 sao, 7 lượt)


## Bài 2: Phân tích đánh giá theo thể loại

In [5]:
# Bước 1: Tạo map (MovieID → [Genre1, Genre2, ...])
movie_genres = movies_rdd.map(lambda line: line.split(",")) \
                          .map(lambda x: (x[0], x[2].split("|")))

# Bước 2: Đọc ratings → (MovieID, rating)
ratings_map = ratings_rdd.map(lambda line: line.split(",")) \
                          .map(lambda x: (x[1], float(x[2])))

# Bước 3: Join ratings với genres
joined = ratings_map.join(movie_genres)
# Kết quả: (MovieID, (rating, [genres]))

# Bước 4: Flat map → (genre, (rating, 1)) cho mỗi thể loại
genre_rating = joined.flatMap(
    lambda x: [(genre, (x[1][0], 1)) for genre in x[1][1]]
)

# Bước 5: Tính trung bình
genre_avg = genre_rating.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])) \
                         .mapValues(lambda x: round(x[0]/x[1], 2)) \
                         .sortBy(lambda x: -x[1])

print("=== Kết quả Bài 2 ===")
print("Điểm trung bình theo thể loại:")
for genre, avg in genre_avg.collect():
    print(f"  {genre}: {avg}")

=== Kết quả Bài 2 ===
Điểm trung bình theo thể loại:
  Film-Noir: 4.36
  Mystery: 4.0
  Horror: 4.0
  Fantasy: 3.86
  Crime: 3.81
  Drama: 3.76
  Sci-Fi: 3.73
  Action: 3.71
  Thriller: 3.7
  Family: 3.67
  Adventure: 3.63
  Biography: 3.56


## Bài 3: Phân tích đánh giá theo giới tính

In [6]:
# Bước 1: Đọc users.txt → (UserID, Gender)
users_rdd = sc.textFile("users.txt")
user_gender = users_rdd.map(lambda line: line.split(",")) \
                        .map(lambda x: (x[0], x[1]))  # (UserID, 'M'/'F')

# Bước 2: Ratings → (UserID, (MovieID, rating))
ratings_user = ratings_rdd.map(lambda line: line.split(",")) \
                           .map(lambda x: (x[0], (x[1], float(x[2]))))

# Bước 3: Join để thêm gender
joined = ratings_user.join(user_gender)
# Kết quả: (UserID, ((MovieID, rating), gender))

# Bước 4: Map → ((MovieID, gender), (rating, 1))
movie_gender_rating = joined.map(
    lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1], 1))
)

# Bước 5: Tính trung bình
result = movie_gender_rating.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])) \
                             .mapValues(lambda x: round(x[0]/x[1], 2)) \
                             .sortBy(lambda x: x[0][0])

print("=== Kết quả Bài 3 ===")
print("Điểm TB mỗi phim theo giới tính (MovieID, Gender): avg")
for (movie_id, gender), avg in result.take(15):
    title = movie_map.get(movie_id, movie_id)
    print(f"  {title} | {gender}: {avg}")

=== Kết quả Bài 3 ===
Điểm TB mỗi phim theo giới tính (MovieID, Gender): avg
  Lawrence of Arabia (1962) | M: 3.55
  Lawrence of Arabia (1962) | F: 3.31
  Psycho (1960) | F: 4.0
  The Godfather: Part II (1974) | M: 4.06
  The Godfather: Part II (1974) | F: 3.94
  Sunset Boulevard (1950) | F: 4.5
  Sunset Boulevard (1950) | M: 4.33
  E.T. the Extra-Terrestrial (1982) | M: 3.81
  E.T. the Extra-Terrestrial (1982) | F: 3.55
  The Terminator (1984) | F: 4.14
  The Terminator (1984) | M: 3.93
  Fight Club (1999) | M: 3.5
  Fight Club (1999) | F: 3.5
  The Silence of the Lambs (1991) | M: 3.33
  The Silence of the Lambs (1991) | F: 3.0


## Bài 4: Phân tích đánh giá theo nhóm tuổi

In [7]:
# Hàm phân nhóm tuổi
def age_group(age):
    age = int(age)
    if age < 18:   return "<18"
    elif age < 30: return "18-29"
    elif age < 45: return "30-44"
    elif age < 60: return "45-59"
    else:          return "60+"

# Bước 1: (UserID, AgeGroup)
user_agegroup = users_rdd.map(lambda line: line.split(",")) \
                          .map(lambda x: (x[0], age_group(x[2])))

# Bước 2: Ratings → (UserID, (MovieID, rating))
ratings_user = ratings_rdd.map(lambda line: line.split(",")) \
                           .map(lambda x: (x[0], (x[1], float(x[2]))))

# Bước 3: Join
joined = ratings_user.join(user_agegroup)
# (UserID, ((MovieID, rating), agegroup))

# Bước 4: ((MovieID, agegroup), (rating, 1))
movie_age_rating = joined.map(
    lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1], 1))
)

# Bước 5: Tính trung bình
result = movie_age_rating.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])) \
                          .mapValues(lambda x: round(x[0]/x[1], 2)) \
                          .sortBy(lambda x: x[0])

print("=== Kết quả Bài 4 ===")
print("Điểm TB mỗi phim theo nhóm tuổi:")
for (movie_id, age_grp), avg in result.take(15):
    title = movie_map.get(movie_id, movie_id)
    print(f"  {title} | {age_grp}: {avg}")

=== Kết quả Bài 4 ===
Điểm TB mỗi phim theo nhóm tuổi:
  Lawrence of Arabia (1962) | 18-29: 3.33
  Lawrence of Arabia (1962) | 30-44: 3.41
  Lawrence of Arabia (1962) | 45-59: 3.62
  Psycho (1960) | 30-44: 4.0
  The Godfather: Part II (1974) | 18-29: 3.7
  The Godfather: Part II (1974) | 30-44: 4.12
  Sunset Boulevard (1950) | 18-29: 4.0
  Sunset Boulevard (1950) | 30-44: 4.5
  Sunset Boulevard (1950) | 45-59: 4.5
  E.T. the Extra-Terrestrial (1982) | 18-29: 3.4
  E.T. the Extra-Terrestrial (1982) | 30-44: 3.82
  E.T. the Extra-Terrestrial (1982) | 45-59: 4.0
  E.T. the Extra-Terrestrial (1982) | 60+: 3.0
  The Terminator (1984) | 18-29: 4.38
  The Terminator (1984) | 30-44: 3.94


## Bài 5: Phân tích đánh giá theo nghề nghiệp

In [8]:
# Đọc occupation.txt → dict {OccupationID: OccupationName}
occ_rdd = sc.textFile("occupation.txt")
occ_map = occ_rdd.map(lambda line: line.split(",")) \
                  .map(lambda x: (x[0], x[1])) \
                  .collectAsMap()

# Bước 1: users.txt → (UserID, OccupationName)
user_occ = users_rdd.map(lambda line: line.split(",")) \
                     .map(lambda x: (x[0], occ_map.get(x[3], "Unknown")))

# Bước 2: Ratings → (UserID, rating)
ratings_user2 = ratings_rdd.map(lambda line: line.split(",")) \
                            .map(lambda x: (x[0], float(x[2])))

# Bước 3: Join → gán occupation vào từng rating
joined = ratings_user2.join(user_occ)
# (UserID, (rating, occupation))

# Bước 4: (occupation, (rating, 1))
occ_rating = joined.map(lambda x: (x[1][1], (x[1][0], 1)))

# Bước 5: Tính tổng và trung bình
result = occ_rating.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])) \
                    .mapValues(lambda x: (round(x[0]/x[1], 2), x[1])) \
                    .sortBy(lambda x: -x[1][0])

print("=== Kết quả Bài 5 ===")
print("Trung bình rating và số lượt theo nghề nghiệp:")
for occ, (avg, count) in result.collect():
    print(f"  {occ}: avg={avg}, total_ratings={count}")

=== Kết quả Bài 5 ===
Trung bình rating và số lượt theo nghề nghiệp:
  Programmer: avg=4.25, total_ratings=10
  Designer: avg=4.0, total_ratings=13
  Student: avg=4.0, total_ratings=8
  Consultant: avg=3.86, total_ratings=14
  Nurse: avg=3.86, total_ratings=11
  Journalist: avg=3.85, total_ratings=17
  Artist: avg=3.73, total_ratings=11
  Teacher: avg=3.7, total_ratings=5
  Doctor: avg=3.69, total_ratings=21
  Salesperson: avg=3.65, total_ratings=17
  Lawyer: avg=3.65, total_ratings=17
  Accountant: avg=3.58, total_ratings=6
  Engineer: avg=3.56, total_ratings=18
  Manager: avg=3.47, total_ratings=16


## Bài 6: Phân tích đánh giá theo thời gian (năm)

In [9]:
import datetime

# Hàm chuyển Unix timestamp → năm
def timestamp_to_year(ts):
    return datetime.datetime.fromtimestamp(int(ts)).year

# Bước 1: Đọc ratings → (year, (rating, 1))
year_rating = ratings_rdd.map(lambda line: line.split(",")) \
                          .map(lambda x: (timestamp_to_year(x[3]), (float(x[2]), 1)))

# Bước 2: Reduce → tổng điểm và số lượt mỗi năm
result = year_rating.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])) \
                     .mapValues(lambda x: (x[1], round(x[0]/x[1], 2))) \
                     .sortByKey()

print("=== Kết quả Bài 6 ===")
print("Năm | Tổng lượt đánh giá | Điểm TB")
for year, (count, avg) in result.collect():
    print(f"  {year}: {count} lượt, avg={avg}")

=== Kết quả Bài 6 ===
Năm | Tổng lượt đánh giá | Điểm TB
  2020: 184 lượt, avg=3.75


In [ ]:
# Dừng SparkContext khi xong
sc.stop()
print("Done!")